# Classification de Races de Chiens
# Authors CHAKER Zakaria NASNGUE Nathan
## Stanford Dogs Dataset — 20 580 images, 120 races
Dans ce notebook nous verrons les différentes étapes qui menerons à la classification des races de chiens.

0.Initialisation des bibliotheques.
1. Téléchargement et organisation du dataset
2. Exploration et visualisation des données -> à des fins statistiques pour évaluer les données présentes dans le DATA SET  (redimensionnement ?, nombre d'images par sous dossier, etc.)
3. Préparation des images
4. Entraînement d'un modèle de classification
5. Évaluation et résultats

Notes
- L'entraînement complet peut prendre plusieurs heures.

## 0. Installation des bibliothèques

In [ ]:
# Commande pour installer les bibliothèques nécessaires à l'entrainement du modèle
pip install numpy pandas matplotlib seaborn scikit-learn opencv-python Pillow tqdm

## 1. Téléchargement du Dataset Stanford Dogs

In [ ]:
import os
import urllib.request
import tarfile
import subprocess
import sys

# Author CHAKER Zakaria & NSANGUE Nathan
# Dans cette section nous récupérons le dataset présent en ligne...
# Ce script permet de télécharger et d'extraire le dataset en utilisant des bibliothèques adaptées
# comme tarfile pour l'extraction.
# Une vérification de la connexion Internet est effectuée avant le téléchargement.
# Gestion des erreurs ajoutée pour améliorer la robustesse.
# Date 10/03/2026
# Nom du script: recuperation_dataset.py
# Version 1.1

BASE_DIR = "./stanford_dogs"
os.makedirs(BASE_DIR, exist_ok=True)

# Chargement des URL dans des variables 
IMAGES_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
ANNOTS_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar"
LISTS_URL  = "http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar"

def check_internet():
    """
    Vérifie la connexion Internet en envoyant un ping vers 8.8.8.8.
    Retourne True si Internet est accessible, sinon False.
    """
    try:
        param = "-n" if sys.platform.lower() == "win32" else "-c"
        result = subprocess.run(
            ["ping", param, "1", "8.8.8.8"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return result.returncode == 0
    except Exception as e:
        print("Erreur lors de la vérification de la connexion Internet :", e)
        return False

def download_and_extract(url, dest_dir):
    """
    Télécharge et décompresse un fichier .tar
    """
    filename = os.path.join(dest_dir, url.split("/")[-1])

    # Téléchargement
    if not os.path.exists(filename):
        print(f"Téléchargement de {url.split('/')[-1]}...")
        try:
            urllib.request.urlretrieve(url, filename)
            print("Téléchargement terminé.")
        except Exception as e:
            print("Erreur lors du téléchargement :", e)
            return
    else:
        print(f"{url.split('/')[-1]} déjà présent, on passe.")

    # Extraction des archives
    print("Extraction en cours...")
    try:
        with tarfile.open(filename) as tar:
            tar.extractall(dest_dir)
        print("Extraction terminée.")
    except Exception as e:
        print("Erreur lors de l'extraction :", e)

# Vérification de la connexion Internet...
if not check_internet():
    print("Aucune connexion Internet détectée. Veuillez vérifier votre réseau.")
    sys.exit(1)

download_and_extract(IMAGES_URL, BASE_DIR)
download_and_extract(LISTS_URL, BASE_DIR)

print("\nDataset prêt.")

## 2. Exploration du Dataset

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# -------------------------------------------------------------------------
# Author CHAKER Zakaria & NSANGUE Nathan
# Objectif :
# Analyse exploratoire du dataset Stanford Dogs....
#
# Cette partie du code permet de :
# - Vérifier l'existence du dossier contenant les images.
# - Identifier les classes (races de chiens) présentes dans le dataset.
# - Compter le nombre d'images par classe.
# - Générer des statistiques globales (total, moyenne, min, max) relatives au Dataset
# - Analyser les dimensions des images (largeur et hauteur)
#
# Sorties :
# - distribution_races.png : répartition des images par race
# - dimensions_images.png : distribution des tailles d’images
#
# Remarque :
# Cette étape est essentielle pour comprendre la structure du dataset avant l’entraînement du modèle (équilibrage, taille des images, etc.).
# Date 15/03/2026
# Version 1.1
# -------------------------------------------------------------------------

BASE_DIR = "./stanford_dogs"
IMAGES_DIR = os.path.join(BASE_DIR, "Images")

# Vérification de l'existence du dossier Images
if not os.path.exists(IMAGES_DIR):
    print("Le dossier Images est introuvable. Vérifiez que le dataset est bien téléchargé.")
    exit(1)

# Lister toutes les races (= sous-dossiers)
try:
    breed_folders = sorted([
        d for d in os.listdir(IMAGES_DIR)
        if os.path.isdir(os.path.join(IMAGES_DIR, d))
    ])
except Exception as e:
    print("Erreur lors de la lecture du dossier Images :", e)
    exit(1)

# Affichage des résultats
print(f"Nombre de races : {len(breed_folders)}")

breed_counts = {}
for folder in breed_folders:
    path = os.path.join(IMAGES_DIR, folder)
    images = glob.glob(os.path.join(path, "*.jpg"))
    name = folder.split("-", 1)[-1].replace("_", " ").title()
    breed_counts[name] = len(images)

df_counts = pd.DataFrame.from_dict(breed_counts, orient="index", columns=["nb_images"])
df_counts = df_counts.sort_values("nb_images", ascending=False)

print(" Statistiques globales :")
print(f"  Total images     : {df_counts['nb_images'].sum()}")
print(f"  Races            : {len(df_counts)}")
print(f"  Moy. par race    : {df_counts['nb_images'].mean():.1f}")
print(f"  Min. par race    : {df_counts['nb_images'].min()}")
print(f"  Max. par race    : {df_counts['nb_images'].max()}")

# Visualisation : distribution du nombre d'images par race
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Top 20 races les plus représentées
top20 = df_counts.head(20)
axes[0].barh(top20.index[::-1], top20["nb_images"][::-1], color="steelblue")
axes[0].set_title("Top 20 races les plus représentées", fontsize=13)
axes[0].set_xlabel("Nombre d'images")
axes[0].axvline(df_counts["nb_images"].mean(), color="red", linestyle="--", label="Moyenne")
axes[0].legend()

# Distribution globale (affichage en histogramme)
axes[1].hist(df_counts["nb_images"], bins=20, color="coral", edgecolor="white")
axes[1].set_title("Distribution du nb d'images par race", fontsize=13)
axes[1].set_xlabel("Nombre d'images")
axes[1].set_ylabel("Nombre de races")
axes[1].axvline(df_counts["nb_images"].mean(), color="red", linestyle="--", label="Moyenne")
axes[1].legend()

plt.tight_layout()
plt.savefig("distribution_races.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graphique sauvegarde : distribution_races.png")

print("Analyse des dimensions (echantillon 500 images)...")
widths, heights = [], []

all_images = glob.glob(os.path.join(IMAGES_DIR, "**", "*.jpg"), recursive=True)
sample_imgs = np.random.choice(all_images, min(500, len(all_images)), replace=False)

for img_path in sample_imgs:
    try:
        with Image.open(img_path) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
    except:
        pass

print(f"\nDimensions des images :")
print(f"  Largeur  — min: {min(widths)}, max: {max(widths)}, moy: {np.mean(widths):.0f}")
print(f"  Hauteur  — min: {min(heights)}, max: {max(heights)}, moy: {np.mean(heights):.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30, color="teal", edgecolor="white")
axes[0].set_title("Distribution des largeurs")
axes[0].set_xlabel("Pixels")
axes[1].hist(heights, bins=30, color="orange", edgecolor="white")
axes[1].set_title("Distribution des hauteurs")
axes[1].set_xlabel("Pixels")
plt.tight_layout()
plt.savefig("dimensions_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Préparation des Données
- Redimensionner toutes les images à **128×128** 
- Normaliser les pixels entre 0 et 1
- Séparer en train / validation / test
- Créer les labels (encodage numérique des races)

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

# -------------------------------------------------------------------------
# Author : CHAKER Zakaria & NSANGUE Nathan
# Date   :16/03/2026
# Version: 1.2
#
# Objectif :
# Redimensionnement des images.
#
# Cette étape permet de :
# - Parcourir l’ensemble des classes (races de chiens)
# - Limiter le nombre d’images par classe (MAX_PER_CLASS)
# - Redimensionner toutes les images à une taille uniforme (128 x 128)
# - Sauvegarder les images prétraitees au format JPG dans un nouveau dossier (resized/)
#
# Sortie :
# - Dossier "resized/" contenant les images redimensionnées -> organisées par race (même structure que le dataset original)
# -------------------------------------------------------------------------

IMG_SIZE = 128   # Taille cible (128x128 pixels)
MAX_PER_CLASS = 80  # ombre maxi des echantillons à redimensioner

BASE_DIR = "./stanford_dogs"
IMAGES_DIR = os.path.join(BASE_DIR, "Images")

# Vérification de l'existence du dossier Images /*Prerequis*/
if not os.path.exists(IMAGES_DIR):
    print("Le dossier Images est introuvable. Vérifiez que le dataset est bien téléchargé.")
    exit(1)     

RESIZED_DIR = os.path.join(BASE_DIR, "resized")
os.makedirs(RESIZED_DIR, exist_ok=True)

try:
    breed_folders = sorted([
        d for d in os.listdir(IMAGES_DIR)
        if os.path.isdir(os.path.join(IMAGES_DIR, d))
    ])
except Exception as e:
    print("Erreur lors de la lecture du dossier Images :", e)
    exit(1)              


for folder in tqdm(breed_folders, desc="Traitement par race"):
    path = os.path.join(IMAGES_DIR, folder)
    imgs = glob.glob(os.path.join(path, "*.jpg"))
    
    # Limite MAX_PER_CLASS
    if MAX_PER_CLASS is not None:
        imgs = imgs[:MAX_PER_CLASS]
    
    # Créer dessous-dossier pour enregistrer les images redimensionnées
    save_dir = os.path.join(RESIZED_DIR, folder)
    os.makedirs(save_dir, exist_ok=True)
    
    for img_path in imgs:
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE))
            
            # Nouveau nom de fichier
            save_path = os.path.join(save_dir, os.path.basename(img_path))
            
            # Savig en JPG
            img.save(save_path, format="JPEG", quality=95)
            
        except Exception as e:
            print(f"Erreur sur {img_path} : {e}")

In [ ]:
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

# -------------------------------------------------------------------------
# Author : CHAKER Zakaria & NSANGUE Nathan
# Date   : 16/03/2026
# Version: 1.0
#
# Objectif :
# Préparer les données à partir du dossier "resized/".
#
# Cette étape permet de :
# - Lire toutes les images redimensionnées
# - Associer chaque image à sa classe (race)
# - Encoder les labels (texte en entiers)
# - Mélanger les données
# - Decoupage en train/validation/test
#
# Sortie :
# - X_train, X_val, X_test : chemins des images
# - y_train, y_val, y_test : labels encodés
# -------------------------------------------------------------------------

BASE_DIR = "./stanford_dogs"
RESIZED_DIR = os.path.join(BASE_DIR, "resized")

# Vérification
if not os.path.exists(RESIZED_DIR):
    print("Le dossier resized est introuvable. Lance d'abord le script de redimensionnement.")
    exit(1)

# Récupération des classes
breed_folders = sorted([
    d for d in os.listdir(RESIZED_DIR)
    if os.path.isdir(os.path.join(RESIZED_DIR, d))
])

# Création des données (chemins + labels)
image_paths = []
labels = []

for folder in breed_folders:
    path = os.path.join(RESIZED_DIR, folder)
    imgs = glob.glob(os.path.join(path, "*.jpg"))
    
    for img_path in imgs:
        image_paths.append(img_path)
        labels.append(folder.split("-", 1)[-1])

image_paths = np.array(image_paths)
labels = np.array(labels)

print(f"\nNombre total d'images : {len(image_paths)}")

# Encodage des labels
le = LabelEncoder() #Encodage des labels avec Classe LabelEncoder
y_encoded = le.fit_transform(labels)

# Mélange des données
image_paths, y_encoded = shuffle(image_paths, y_encoded, random_state=42)


# Decoupage 70% train / 15% validation / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(image_paths, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Affichage
print("\nRepartition des donnees :")
print(f"Train : {len(X_train)} images ({len(X_train)/len(image_paths)*100:.0f}%)")
print(f"Validation : {len(X_val)} images ({len(X_val)/len(image_paths)*100:.0f}%)")
print(f"Test : {len(X_test)} images ({len(X_test)/len(image_paths)*100:.0f}%)")

## 4. Modèle de Classification

Baseline rapide** — Random Forest sur HOG features (adapté CPU)

In [ ]:
import numpy as np
from tqdm import tqdm
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import time

# -------------------------------------------------------------------------
# Fonction : extract_hog_features
# Objectif  : Extraire les features HOG de chaque image
# -------------------------------------------------------------------------
data = np.load("dataset.npz", allow_pickle=True)
X_train = data["X_train"]
X_val   = data["X_val"]
X_test  = data["X_test"]
y_train = data["y_train"]
y_val   = data["y_val"]
y_test  = data["y_test"]



def extract_hog_features(image_paths, desc="HOG"):
    """Extrait les features HOG de chaque image."""
    features = []
    for path in tqdm(image_paths, desc=desc):
        try:
            img = Image.open(path).convert("RGB")  # Ouvre et convertit en RGB
            img = np.array(img)                    # Convertit en array (uint8 0-255)
            feat = hog(
                img,
                orientations=12,
                pixels_per_cell=(16, 16),
                cells_per_block=(2, 2),
                channel_axis=-1
            )
            features.append(feat)
        except Exception as e:
            print(f"Erreur sur {path}: {e}")
    return np.array(features)

print("Extraction des features HOG (train)...")
X_train_hog = extract_hog_features(X_train, "Train HOG")

print("Extraction des features HOG (val)...")
X_val_hog = extract_hog_features(X_val, "Val HOG")

print("Extraction des features HOG (test)...")
X_test_hog = extract_hog_features(X_test, "Test HOG")

print(f"\nShape features HOG : {X_train_hog.shape}")

print("=" * 50)
print("Modèle A — Random Forest (Baseline)")
print("=" * 50)

t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=100,    # nombre d'arbres
    max_depth=50,        # profondeur maximale des arbres
    n_jobs=-1,           # utilise tous les cœurs CPU
    random_state=42
)

# Entraînement
rf.fit(X_train_hog, y_train)
t1 = time.time()

# Prédiction sur la validation
y_pred_rf = rf.predict(X_val_hog)
acc_rf = accuracy_score(y_val, y_pred_rf)

print(f"  Temps d'entraînement : {t1 - t0:.1f}s")
print(f"  Accuracy (validation) : {acc_rf * 100:.1f}%")

## 5. Évaluation et Résultats

In [ ]:
# Matrice de confusion (top 20 races)
top20_breeds = df_report.head(20).index.tolist()
top20_idx    = [list(le.classes_).index(b) for b in top20_breeds if b in list(le.classes_)]

mask = np.isin(y_test, top20_idx)
cm   = confusion_matrix(y_test[mask], y_pred_test[mask], labels=top20_idx)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=False, fmt="d", cmap="Blues",
    xticklabels=[b[:15] for b in top20_breeds],
    yticklabels=[b[:15] for b in top20_breeds],
    ax=ax
)
ax.set_title("Matrice de Confusion — Top 20 races", fontsize=14)
ax.set_xlabel("Prédit")
ax.set_ylabel("Réel")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Matrice sauvegardée : confusion_matrix.png")

## 6. Sauvegarde du Modèle

In [ ]:
import pickle

# Sauvegarder le meilleur modèle + le label encoder
with open("best_model.pkl", "wb") as f:
    pickle.dump({"model": best_model, "label_encoder": le}, f)

print(f"Modele sauvegarde : best_model.pkl")
print(f"Pour recharger : pickle.load(open('best_model.pkl','rb'))")

## 7. Utiliser le Modèle sur une Nouvelle Image

In [ ]:
def predict_breed(image_path, model, label_encoder, img_size=128):
    """
    Prédit la race d'un chien à partir du chemin d'une image.
    
    Args:
        image_path : chemin vers l'image (ex: 'mon_chien.jpg')
        model      : modèle entraîné
        label_encoder : LabelEncoder utilisé pendant l'entraînement
    
    Returns:
        Nom de la race prédite
    """
    # Charger et préparer l'image
    img = Image.open(image_path).convert("RGB")
    img = img.resize((img_size, img_size))
    img_array = np.array(img, dtype=np.float32) / 255.0
    
    # Extraire les features HOG
    img_uint8 = (img_array * 255).astype(np.uint8)
    features = hog(img_uint8, orientations=8,
                   pixels_per_cell=(16, 16), cells_per_block=(2, 2),
                   channel_axis=-1)
    
    # Prédire
    pred_idx  = model.predict([features])[0]
    pred_race = label_encoder.classes_[pred_idx].replace("_", " ").title()
    
    return pred_race

# Exemple d'utilisation :
# race = predict_breed("mon_chien.jpg", best_model, le)
# print(f"Race prédite : {race}")

print("Fonction predict_breed() prête à l'emploi !")
print("Utilise : predict_breed('chemin/vers/image.jpg', best_model, le)")